<!-- field-paper-role-banner -->
> 🟩 **정답·해설본** — 같은 셀 위치의 실행 가능한 참고 구현입니다. 복사하기 전에 수식과 tensor shape를 설명하세요.

# 생성 모델 · 03. DCGAN

## Unsupervised Representation Learning with Deep Convolutional Generative Adversarial Networks

- **원 논문:** [Unsupervised Representation Learning with Deep Convolutional Generative Adversarial Networks](https://arxiv.org/abs/1511.06434)
- **저자 / 발표:** Alec Radford, Luke Metz, and Soumith Chintala · ICLR (2016)
- **난이도 / 예상 시간:** 중급 · 약 60분
- **선수 지식:** convolution, batch normalization, GAN loss
- **로컬 학습 데이터:** `data/field_curriculum/generative_samples.npz`

이 자료는 핵심 메커니즘을 확인하는 **하드웨어 인식 교육용 미니 재현**입니다.
PyTorch 학습은 CUDA → Apple MPS → CPU 순으로 자동 선택하고, NumPy·환경 simulation 같은
CPU 연산은 그대로 유지합니다. 원 논문의 전체 데이터·모델·학습 예산·최종 benchmark를
재현했다는 뜻은 아닙니다.

## 재현 범위

**보존하는 핵심:** 8×8 영상용 작은 DCGAN에서 strided convolution, batch normalization, tanh 설계를 검증한다.

**축소하거나 생략한 부분:** 논문은 LSUN 등 64×64 이미지와 깊은 convolution을 사용한다. 축소판은 §3의 architecture constraints를 8×8에 적용한다.

완료 기준은 정답과 문자가 같은 코드가 아니라, TODO 구현·불변식 assertion·작은 metric 또는
시각화·마지막 해석을 모두 마치는 것입니다.

## 원 논문 ↔ 실습 지도

        절·수식·그림 번호는 위 primary source를 기준으로 합니다. 판본에 따라 페이지가 달라질 수
        있으므로 절 제목과 식 번호를 함께 확인하세요.

        | 원 논문의 위치 | 대응 코드·실험 | 확인할 증거 |
        |---|---|---|
        | §3, bullet list | architecture constraints | pooling 대신 strided convolution, G의 ReLU/tanh와 D의 LeakyReLU를 적용한다. |
| §3 and Figure 1 | generator topology | latent spatial tensor를 fractional-strided convolution으로 확대한다. |
| §4–§5 | training/representation checks | adversarial loss와 생성 샘플을 함께 관찰한다. |

## 핵심 재현

        §3의 설계 규칙을 8×8에 맞춥니다. Generator는 transposed convolution으로 해상도를
        키우고 마지막에 `tanh`, discriminator는 strided convolution과 LeakyReLU를 씁니다.

In [ ]:
from pathlib import Path
import math
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
import torch.nn.functional as F
from llm_engineering_lab.acceleration import get_accelerator

torch.manual_seed(331)
np.random.seed(331)
ACCELERATOR = get_accelerator()
DEVICE = ACCELERATOR.device
# These teaching batches are intentionally tiny; GPU launch/transfer
# overhead can dominate, so use AI_LAB_DEVICE=cpu for minimum wall time.

DATA_PATH = Path("data/field_curriculum/generative_samples.npz")
assert DATA_PATH.exists(), f"준비된 로컬 데이터가 없습니다: {DATA_PATH}"
raw = np.load(DATA_PATH)
expected = {"points", "point_labels", "images", "image_labels", "domain_a", "domain_b"}
assert expected.issubset(raw.files), f"NPZ 키 불일치: {sorted(raw.files)}"
points = torch.tensor(raw["points"], dtype=torch.float32)
point_labels = torch.tensor(raw["point_labels"], dtype=torch.long)
images = torch.tensor(raw["images"], dtype=torch.float32)
image_labels = torch.tensor(raw["image_labels"], dtype=torch.long)
domain_a = torch.tensor(raw["domain_a"], dtype=torch.float32)
domain_b = torch.tensor(raw["domain_b"], dtype=torch.float32)
points, point_labels, images, image_labels, domain_a, domain_b = ACCELERATOR.move(
    points, point_labels, images, image_labels, domain_a, domain_b
)


# Seeded random draws stay on CPU so CUDA, MPS, and CPU follow the same
# sampling stream; complete tensors are then transferred to the lab device.
def randn_device(size, *, generator):
    return torch.randn(size, generator=generator).to(DEVICE)


def rand_device(size, *, generator):
    return torch.rand(size, generator=generator).to(DEVICE)


def randint_device(high, size, *, generator):
    return torch.randint(high, size, generator=generator).to(DEVICE)


if float(images.min()) < 0.0:
    images_01 = ((images + 1.0) / 2.0).clamp(0.0, 1.0)
else:
    images_01 = images.clamp(0.0, 1.0)
assert points.shape == (600, 2) and images.shape == (300, 1, 8, 8)
assert domain_a.shape == domain_b.shape == (600, 2)
print(ACCELERATOR.summary())
print("local data:", tuple(points.shape), tuple(images.shape), tuple(domain_a.shape))

## 포트폴리오 구현 설계: 수식 → 책임 → 검증

논문의 핵심 목적을 실행 가능한 객체의 `forward`, `loss`, `update`, `evaluate`로
나눕니다. 아래 식이 클래스 내부에서 생략되지 않고 어떤 tensor 연산이 되는지
메서드별로 추적해 보세요.

$$
\mathcal L_G=-\mathbb E_z[\log \sigma(D(G(z)))],\qquad \mathcal L_D=\operatorname{BCE}_{real}+\operatorname{BCE}_{fake}
$$

- **기호와 역할:** $G$는 transposed convolution, $D$는 strided convolution, $\sigma$는 logit을 확률로 바꾸는 sigmoid입니다.
- **shape 계약:** `z [N, 8, 1, 1] → image [N, 1, 8, 8] → D logit [N]`
- **논문 위치:** `§3, bullet list`의 **architecture constraints**
- **코드 Task:** DCGAN 설계 규칙과 adversarial update, tanh 출력 범위 검사를 구현합니다.
- **학습·평가 흐름:** [-1,1] image batch → convolutional D/G update → 고정 latent sample 평가
- **원문 대비 한계:** 논문은 LSUN 등 64×64 이미지와 깊은 convolution을 사용한다. 축소판은 §3의 architecture constraints를 8×8에 적용한다.
- **구현 이유:** 수식의 tensor 경계와 optimizer 책임을 클래스 메서드별로
  분리해 코드와 논문을 한 줄씩 대조할 수 있게 합니다.
- **완료 증거:** 마지막 셀에서 `update`, parameter 변화, 평가 metric을
  모두 `assert`합니다.

정답 클래스는 교육용 편의 함수가 핵심 계산을 대신하지 않도록 loss와 update 순서를
메서드 본문에 드러냅니다. 실습본에서는 같은 공개 API를 유지한 채 TODO를 채우세요.

In [ ]:
class ConvolutionalGanLab(nn.Module):
    """DCGAN의 핵심 학습·평가 경로. 입출력 계약: z [N, 8, 1, 1] → image [N, 1, 8, 8] → D logit
    [N]."""

    def __init__(
        self, generator, discriminator, generator_optimizer, discriminator_optimizer
    ):
        """구성 요소와 optimizer를 저장한다. 학습은 update에서만 수행한다."""
        super().__init__()
        self.generator = generator
        self.discriminator = discriminator
        self.generator_optimizer = generator_optimizer
        self.discriminator_optimizer = discriminator_optimizer

    def config(self):
        """재현 범위와 핵심 하이퍼파라미터를 dict로 반환한다."""
        return {"noise_shape": (8, 1, 1), "image_shape": (1, 8, 8)}

    def forward(self, noise):
        """논문 순전파를 계산한다. shape 계약: z [N, 8, 1, 1] → image [N, 1, 8, 8] → D logit [N]."""
        image = self.generator(noise)
        return (image, self.discriminator(image))

    def loss(self, real, noise):
        """논문의 목적 함수를 미분 가능한 scalar tensor로 반환한다."""
        fake, fake_logits = self.forward(noise)
        real_logits = self.discriminator(real)
        discriminator_loss = F.binary_cross_entropy_with_logits(
            real_logits, torch.ones_like(real_logits)
        ) + F.binary_cross_entropy_with_logits(
            fake_logits.detach(), torch.zeros_like(fake_logits)
        )
        generator_loss = F.binary_cross_entropy_with_logits(
            self.discriminator(fake), torch.ones_like(fake_logits)
        )
        return (discriminator_loss, generator_loss)

    def update(self, real, discriminator_noise, generator_noise):
        """zero_grad, backward, step을 수행하고 유한한 float loss를 반환한다."""
        fake = self.generator(discriminator_noise).detach()
        real_logits = self.discriminator(real)
        fake_logits = self.discriminator(fake)
        discriminator_loss = F.binary_cross_entropy_with_logits(
            real_logits, torch.ones_like(real_logits)
        ) + F.binary_cross_entropy_with_logits(
            fake_logits, torch.zeros_like(fake_logits)
        )
        self.discriminator_optimizer.zero_grad()
        discriminator_loss.backward()
        self.discriminator_optimizer.step()
        generated = self.generator(generator_noise)
        generated_logits = self.discriminator(generated)
        generator_loss = F.binary_cross_entropy_with_logits(
            generated_logits, torch.ones_like(generated_logits)
        )
        self.generator_optimizer.zero_grad()
        generator_loss.backward()
        self.generator_optimizer.step()
        return (float(discriminator_loss.detach()), float(generator_loss.detach()))

    def evaluate(self, noise):
        """no_grad 경로에서 논문과 연결된 정량 지표 dict를 반환한다."""
        with torch.no_grad():
            images, logits = self.forward(noise)
        return {
            "sample_shape": tuple(images.shape),
            "absolute_max": float(images.abs().max()),
            "mean_logit": float(logits.mean()),
        }

### 단계 2. 핵심 연산 검증

- **논문의 어느 부분인가:** `§3, bullet list` — **architecture constraints**
- **구현 이유:** pooling 대신 strided convolution, G의 ReLU/tanh와 D의 LeakyReLU를 적용한다.
- **읽을 코드:** 아래 `implementation` 셀에서 중간 tensor의 shape, gradient가 흐르는
  경로와 `detach` 또는 `no_grad` 경계를 확인합니다.

실행 결과만 보지 말고, 이 단계의 입력과 출력이 앞선 수식의 어떤 기호인지 먼저
주석으로 적은 뒤 실습하세요.
- **완료 증거:** 아래 셀의 `assert`와 출력 metric이 shape·loss 계약을 검사합니다.

In [ ]:
class ConvG(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.ConvTranspose2d(8, 16, 4, 1, 0),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.ConvTranspose2d(16, 1, 4, 2, 1),
            nn.Tanh(),
        )

    def forward(self, z):
        return self.net(z)


class ConvD(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 16, 4, 2, 1),
            nn.LeakyReLU(0.2),
            nn.Conv2d(16, 1, 4, 1, 0),
        )

    def forward(self, x):
        return self.net(x).flatten()


G, D = ConvG().to(DEVICE), ConvD().to(DEVICE)
z = torch.zeros(5, 8, 1, 1, device=DEVICE)
assert G(z).shape == (5, 1, 8, 8) and D(G(z)).shape == (5,)
assert isinstance(G.net[-1], nn.Tanh) and any(
    isinstance(m, nn.BatchNorm2d) for m in G.modules()
)

### 단계 3. 학습·업데이트

- **논문의 어느 부분인가:** `§4–§5` — **training/representation checks**
- **구현 이유:** adversarial loss와 생성 샘플을 함께 관찰한다.
- **읽을 코드:** 아래 `training` 셀에서 중간 tensor의 shape, gradient가 흐르는
  경로와 `detach` 또는 `no_grad` 경계를 확인합니다.

실행 결과만 보지 말고, 이 단계의 입력과 출력이 앞선 수식의 어떤 기호인지 먼저
주석으로 적은 뒤 실습하세요.
- **완료 증거:** 아래 셀의 `assert`와 출력 metric이 shape·loss 계약을 검사합니다.

In [ ]:
real_images = images_01 * 2 - 1
og = torch.optim.Adam(G.parameters(), lr=0.003, betas=(0.5, 0.999))
od = torch.optim.Adam(D.parameters(), lr=0.003, betas=(0.5, 0.999))
history = []
rng = torch.Generator().manual_seed(332)

portfolio_model = ConvolutionalGanLab(G, D, og, od)
initial_parameters = torch.cat(
    [parameter.detach().flatten().cpu() for parameter in G.parameters()]
)
for _ in range(50):
    index = randint_device(len(real_images), (64,), generator=rng)
    discriminator_noise = randn_device((64, 8, 1, 1), generator=rng)
    generator_noise = randn_device((64, 8, 1, 1), generator=rng)
    history.append(
        portfolio_model.update(real_images[index], discriminator_noise, generator_noise)
    )
final_parameters = torch.cat(
    [parameter.detach().flatten().cpu() for parameter in G.parameters()]
)
parameter_delta = float((final_parameters - initial_parameters).abs().sum())
assert parameter_delta > 0.0 and np.isfinite(history).all()

### 단계 4. 평가·시각화

        - **논문의 어느 부분인가:** `§4–§5` — **training/representation checks**
        - **구현 이유:** adversarial loss와 생성 샘플을 함께 관찰한다.
        - **읽을 코드:** 아래 `evaluation` 셀에서 중간 tensor의 shape, gradient가 흐르는
          경로와 `detach` 또는 `no_grad` 경계를 확인합니다.

        실행 결과만 보지 말고, 이 단계의 입력과 출력이 앞선 수식의 어떤 기호인지 먼저
        주석으로 적은 뒤 실습하세요.
        - **완료 증거:** 아래 셀의 `assert`와 출력 metric이 shape·loss 계약을 검사합니다.

- **평가 범위:** 이 값은 작은 학습 batch의 메커니즘 진단이며, held-out 생성 품질이나 원 논문의 benchmark 성능으로 해석하지 않습니다.

In [ ]:
evaluation_noise = randn_device(
    (8, 8, 1, 1), generator=torch.Generator().manual_seed(333)
)
portfolio_metrics = portfolio_model.evaluate(evaluation_noise)
assert portfolio_metrics

with torch.no_grad():
    samples = G(
        randn_device((8, 8, 1, 1), generator=torch.Generator().manual_seed(333))
    )
assert samples.shape == (8, 1, 8, 8) and float(samples.abs().max()) <= 1.0001
fig, axes = plt.subplots(2, 5, figsize=(9, 4))
h = np.asarray(history)
axes[0, 0].plot(h[:, 0], label="D")
axes[0, 0].plot(h[:, 1], label="G")
axes[0, 0].legend()
axes[1, 0].axis("off")
for ax, img in zip(axes[:, 1:].ravel(), samples.detach().cpu()):
    ax.imshow(img[0], cmap="gray", vmin=-1, vmax=1)
    ax.axis("off")
fig.tight_layout()
plt.show()

공간 축소·확대를 학습 가능한 strided convolution에 맡겨 네트워크가 자체적인 sampling 변환을 학습하도록 하기 위해서입니다.

### 단계 5. 통합 클래스가 실제 update와 evaluate를 소유하는지 검증

- **논문 위치:** `§3, bullet list`의 **architecture constraints**
- **핵심 식:**

  $$
  \mathcal L_G=-\mathbb E_z[\log \sigma(D(G(z)))],\qquad \mathcal L_D=\operatorname{BCE}_{real}+\operatorname{BCE}_{fake}
  $$

- **입출력 shape:** `z [N, 8, 1, 1] → image [N, 1, 8, 8] → D logit [N]`
- **구현 이유:** 앞 셀은 수식을 작은 연산으로 분해해 확인하고,
  이 셀은 같은 구성 요소를 `ConvolutionalGanLab`에 주입해
  최종 학습·평가 경로를 하나로 묶습니다. pooling 대신 strided convolution, G의 ReLU/tanh와 D의 LeakyReLU를 적용한다.
- **완료 증거:** `update(...)`가 유한한 loss를 반환하고,
  `parameter_delta > 0`이며, `evaluate(...)`의 논문 대응 지표가 유한해야 합니다.
  `config()`만 호출하는 것은 실행 재현으로 보지 않습니다.

In [ ]:
config_contract = portfolio_model.config()
numeric_metrics = [
    float(value)
    for value in portfolio_metrics.values()
    if isinstance(value, (int, float))
]
assert config_contract and len(history) > 0
assert np.isfinite(np.asarray(history, dtype=float)).all()
assert parameter_delta > 0.0
assert numeric_metrics and np.isfinite(numeric_metrics).all()
print(
    {
        "class": type(portfolio_model).__name__,
        "steps": len(history),
        "parameter_delta": parameter_delta,
        "metrics": portfolio_metrics,
    }
)

## 정답 해석 가이드

핵심은 최종 숫자를 외우는 것이 아니라 **원문 위치 → 수식 → 코드 → 반증 가능한 증거**를
연결하는 것입니다. 이 노트북은 다음 원 규모 조건을 재현하지 않았습니다:

> 논문은 LSUN 등 64×64 이미지와 깊은 convolution을 사용한다. 축소판은 §3의 architecture constraints를 8×8에 적용한다.

다음 실험에서는 데이터 크기, 모델 폭, 학습 step 중 하나만 바꾸고 동일 seed에서 비교하세요.